# Uni-MuMER K1_datascale_sft / Kaggle 2xT4

Train mới 2 epoch từ Qwen base với LoRA mới và loss SFT chuẩn. Giữ split gốc cho các nhóm ảnh đã có.
Train mới hoặc resume được chọn ở cell 4.5. RUN_TRAINING mặc định tắt.

In [ ]:
# 1. Cấu hình
import json
import os
import uuid

RUN_TRAINING = False  # Đổi thành True khi đã sẵn sàng train và ghi DagsHub.
STOP_AT_GLOBAL_STEP = 250  # Phiên 2: đặt None để chạy tới hết horizon.
RUN_UUID = globals().get("RUN_UUID") or uuid.uuid4().hex
RUN_VARIANT = "K1_datascale_sft"

PROJECT_DIR = "/kaggle/working/test-unimer"
CONDA_DIR = "/kaggle/working/miniconda"
ENV_DIR = f"{CONDA_DIR}/envs/unimumer"
PYTHON = f"{ENV_DIR}/bin/python"

RUN_DIR = f"/kaggle/working/runs/{RUN_UUID}"
BASE_YAML_CONFIG = "train/Uni-MuMER-train.yaml"
RUNTIME_YAML_CONFIG = f"{RUN_DIR}/runtime.yaml"
OUTPUT_DIR = f"{RUN_DIR}/checkpoints"
SYNC_STATE_PATH = f"{RUN_DIR}/checkpoint_sync_state.json"
BASELINE_DATA_ROOT = "/kaggle/working/baseline-data"
NOTEBOOK_PATH = f"{PROJECT_DIR}/uni-mumer-kaggle-dagshub-banmoi.ipynb"

DAGSHUB_USERNAME = "NhatPot"
EXPERIMENT_NAME = "Uni-MuMER-Qwen2.5-VL-3B"
DATASET_REVISION = "8fcc66de1298524c9e47f542a726090c44532e6e"
BASE_SPLIT_MANIFEST = "train/split_manifests/baseline_8000.json.gz"
BASE_SPLIT_SHA256 = "a3ffe3480ceaafcaebf96f31102e9f97dadd9effc56ca235dc573cb0ff00a11a"

# Các cell Bash dùng chung môi trường Python riêng.
os.environ.pop("PYTHONHOME", None)
os.environ.pop("MLFLOW_TRACKING_PASSWORD", None)
os.environ.update({
    key: str(value) for key, value in {
        "PROJECT_DIR": PROJECT_DIR, "CONDA_DIR": CONDA_DIR,
        "ENV_DIR": ENV_DIR, "PYTHON": PYTHON, "RUN_DIR": RUN_DIR,
        "RUN_UUID": RUN_UUID, "RUN_TRAINING": int(RUN_TRAINING),
        "STOP_AT_GLOBAL_STEP": "" if STOP_AT_GLOBAL_STEP is None else STOP_AT_GLOBAL_STEP,
        "BASE_YAML_CONFIG": BASE_YAML_CONFIG, "RUNTIME_YAML_CONFIG": RUNTIME_YAML_CONFIG,
        "OUTPUT_DIR": OUTPUT_DIR, "SYNC_STATE_PATH": SYNC_STATE_PATH,
        "NOTEBOOK_PATH": NOTEBOOK_PATH, "YAML_CONFIG": "",
        "PYTHONNOUSERSITE": 1, "PYTHONPATH": PROJECT_DIR,
        "MLFLOW_TRACKING_URI": f"https://dagshub.com/{DAGSHUB_USERNAME}/test-unimer.mlflow",
        "MLFLOW_TRACKING_USERNAME": DAGSHUB_USERNAME,
        "EXPERIMENT_VARIANT": RUN_VARIANT,
        "MLFLOW_EXPERIMENT_NAME": EXPERIMENT_NAME, "MLFLOW_FLATTEN_PARAMS": "TRUE",
        "MLFLOW_TAGS": json.dumps({"run_uuid": RUN_UUID, "source": "kaggle", "task": "sft", "variant": RUN_VARIANT}),
    }.items()
})
env_bin = f"{ENV_DIR}/bin"
os.environ["PATH"] = os.pathsep.join(
    [env_bin] + [p for p in os.environ.get("PATH", "").split(os.pathsep) if p != env_bin]
)

if RUN_TRAINING:
    from kaggle_secrets import UserSecretsClient
    os.environ["MLFLOW_TRACKING_PASSWORD"] = UserSecretsClient().get_secret("DAGSHUB_TOKEN")

print("Run:", RUN_UUID)
print("Output:", OUTPUT_DIR)


In [ ]:
%%bash
# 2. Tạo môi trường Python 3.10
set -euo pipefail

if [[ ! -x "$CONDA_DIR/bin/conda" ]]; then
  wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
  bash /tmp/miniconda.sh -b -p "$CONDA_DIR"
  rm -f /tmp/miniconda.sh
fi

if [[ ! -x "$PYTHON" ]]; then
  "$CONDA_DIR/bin/conda" tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
  "$CONDA_DIR/bin/conda" tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
  "$CONDA_DIR/bin/conda" create -n unimumer python=3.10 -y
fi

"$PYTHON" --version


In [ ]:
%%bash
# 3. Lấy source cải tiến dev-caitien
set -euo pipefail

if [[ ! -d "$PROJECT_DIR/.git" ]]; then
  git clone --branch dev-caitien --single-branch https://github.com/NhatPot/test-unimer.git "$PROJECT_DIR"
fi
[[ "$(git -C "$PROJECT_DIR" branch --show-current)" == dev-caitien ]] \
  || { echo "Repo hiện có không ở dev-caitien; giữ nguyên, chọn đúng checkout trước."; exit 1; }
git -C "$PROJECT_DIR" rev-parse --short HEAD


In [ ]:
# %%bash
# # 3.5 Download test data
# set -e

# cd "$PROJECT_DIR"

# pip install -q gdown

# gdown "1uHT7iOFHGASc9_HB-PhpIuJizlOUOg1L" -O crohme_test_images.zip

# unzip -q crohme_test_images.zip -d "$PROJECT_DIR"

# rm crohme_test_images.zip

In [ ]:
%%bash
# 4. Cài dependency trong Python riêng
set -euo pipefail
cd "$PROJECT_DIR"

"$PYTHON" --version
"$PYTHON" -m pip install -q -r requirements.txt -e train/LLaMA-Factory
"$PYTHON" -m pip check
"$PYTHON" - <<'PY'
import torch
assert torch.cuda.is_available(), "Chưa nhận CUDA"
names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print("GPU:", names)
assert len(names) == 2 and all("T4" in name for name in names), "Cấu hình này dành cho 2 GPU T4"
PY


In [ ]:
# 4.5 Resume từ DagsHub (tùy chọn, như notebook v5)
import subprocess
from pathlib import Path

USE_DAGSHUB_RESUME = False  # True: full resume; False: train mới.
RESUME_RUN_ID = ""
RESUME_ARTIFACT_PATH = ""
RESUME_SPLIT_SHA256 = ""  # SHA256 của split K1 tạo ở phiên đầu.
RESUME_TOTAL_EPOCHS = 2  # Tổng đích của K1.
RESUME_CHECKPOINT = None
os.environ.pop("RESUME_CHECKPOINT", None)
os.environ["RESUME_BEST_CHECKPOINT"] = ""
os.environ["RESUME_PREPARED_OUTPUT"] = ""
os.environ["YAML_CONFIG"] = ""

if USE_DAGSHUB_RESUME:
    if not RUN_TRAINING:
        raise RuntimeError("Resume cần RUN_TRAINING=True để đọc DagsHub Secrets.")
    if not (RESUME_RUN_ID and RESUME_ARTIFACT_PATH and RESUME_SPLIT_SHA256):
        raise RuntimeError("Resume cần MLflow run ID, checkpoint và SHA256 của split K1.")
    state_path = f"{RUN_DIR}/resume_checkpoint.json"
    subprocess.run(
        [PYTHON, "scripts/dagshub_checkpoint.py",
         "--project-dir", PROJECT_DIR, "--output-dir", f"{RUN_DIR}/resume-source",
         "--run-id", RESUME_RUN_ID, "--artifact-path", RESUME_ARTIFACT_PATH,
         "--state-path", state_path, "--world-size", "2", "--fp16",
         "--manifest-artifact", "training/artifact_manifest.json",
         "--training-output-dir", OUTPUT_DIR],
        cwd=PROJECT_DIR, check=True,
    )
    resume_state = json.loads(Path(state_path).read_text())
    RESUME_CHECKPOINT = resume_state["checkpoint"]
    os.environ["RESUME_BEST_CHECKPOINT"] = resume_state.get("best_checkpoint", "")
    os.environ["RESUME_PREPARED_OUTPUT"] = resume_state["prepared_output"]
    print("Resume:", RESUME_CHECKPOINT, "| Tổng epoch:", RESUME_TOTAL_EPOCHS)
else:
    print("Resume OFF: train mới, giữ cơ chế resume cho lần sau.")


In [ ]:
# 5. Thông số K1

os.environ["YAML_CONFIG"] = ""
YAML_OVERRIDES = {
    "dataset": "parquet_crohme_train, parquet_crohme_train_can, parquet_crohme_train_tree, parquet_crohme_train_error_find, parquet_crohme_train_error_fix, parquet_hme100k_train, parquet_mathwriting_train",
    "max_samples": 8000,  # Cap mặc định trước khi chia tập.
    "max_samples_by_source": {
        "parquet_hme100k_train": 16000,
        "parquet_mathwriting_train": 16000,
    },
    "val_size": 0.1,      # 10% nhóm ảnh; source không chia lại theo dòng.
    "seed": 42,
    "model_revision": "66285546d2b821cf421d4f5eb2576359d3770cd3",
    "use_hard_example_loss": False,
    "hard_example_loss_alpha": 0.5,
    "hard_example_loss_max_weight": 1.5,
    "num_train_epochs": 2,
    "per_device_train_batch_size": 4,
    "gradient_accumulation_steps": 32,
    "learning_rate": 1.0e-4,
    "lora_rank": 64,
    "bf16": False,
    "fp16": True,
    "logging_steps": 1,
    "save_steps": 50,
    "eval_steps": 50,
    "save_total_limit": 2,
    "save_only_model": False,
    "load_best_model_at_end": True,  # Nạp best theo val loss; checkpoint cuối dùng resume.
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False,
    "output_dir": OUTPUT_DIR,
    "overwrite_output_dir": False,
}
if USE_DAGSHUB_RESUME:
    if not RESUME_CHECKPOINT:
        raise RuntimeError("Chạy cell Resume thành công trước khi tạo YAML.")
    YAML_OVERRIDES["num_train_epochs"] = RESUME_TOTAL_EPOCHS
    resume_args = ["--resume-checkpoint", RESUME_CHECKPOINT,
                   "--expected-manifest-sha256", RESUME_SPLIT_SHA256]
else:
    resume_args = []
DATASET_DIR = f"{BASELINE_DATA_ROOT}/{RUN_VARIANT}"
result = subprocess.run(
    [PYTHON, "-m", "scripts.prepare_baseline_data",
     "--project-dir", PROJECT_DIR, "--output-dir", DATASET_DIR,
     "--base-yaml", BASE_YAML_CONFIG, "--runtime-yaml", RUNTIME_YAML_CONFIG,
     "--dataset-revision", DATASET_REVISION,
     "--base-manifest", BASE_SPLIT_MANIFEST, "--base-manifest-sha256", BASE_SPLIT_SHA256,
     "--overrides-json", json.dumps(YAML_OVERRIDES), *resume_args],
    cwd=PROJECT_DIR, capture_output=True, text=True,
)
if result.returncode:
    print(result.stderr)
    raise RuntimeError("Không chuẩn bị được dữ liệu/YAML; xem lỗi phía trên.")
config = json.loads(result.stdout)
YAML_CONFIG, OUTPUT_DIR = config["yaml_config"], config["output_dir"]
tags = json.loads(os.environ["MLFLOW_TAGS"])
tags.update(variant=RUN_VARIANT, split_sha256=config["manifest_sha256"],
            parent_run_id=RESUME_RUN_ID if USE_DAGSHUB_RESUME else "",
            resume_artifact_path=RESUME_ARTIFACT_PATH if USE_DAGSHUB_RESUME else "",
            initial_global_step=str(config["initial_global_step"]),
            initial_epoch=str(config["initial_epoch"]))
os.environ.update(YAML_CONFIG=YAML_CONFIG, OUTPUT_DIR=OUTPUT_DIR,
                  DATASET_DIR=DATASET_DIR, SPLIT_MANIFEST=config["manifest_path"],
                  MLFLOW_TAGS=json.dumps(tags))
print(json.dumps({"samples": config["counts"], "config": config["summary"]}, indent=2))


In [ ]:
%%bash
# 6. Kiểm tra quyền ghi DagsHub (tạo rồi xóa một run thử)
set -euo pipefail
[[ "$RUN_TRAINING" == 1 ]] || { echo "Bỏ qua DagsHub: RUN_TRAINING=False"; exit 0; }
cd "$PROJECT_DIR"
"$PYTHON" scripts/dagshub_logger.py check --experiment "$MLFLOW_EXPERIMENT_NAME"


In [ ]:
%%bash
# 7. Training + sync checkpoint bằng wrapper cũ
set -euo pipefail
[[ "$RUN_TRAINING" == 1 ]] || { echo "Bỏ qua train: RUN_TRAINING=False"; exit 0; }
cd "$PROJECT_DIR"
test -n "$YAML_CONFIG"
[[ ! -e "$OUTPUT_DIR" ]] || { echo "Output đã tồn tại; dùng run mới để tránh ghi đè."; exit 1; }

best_args=()
if [[ -n "${RESUME_PREPARED_OUTPUT:-}" ]]; then
  best_args=(--resume-prepared-output "$RESUME_PREPARED_OUTPUT")
fi

MPLBACKEND=Agg "$PYTHON" scripts/train_with_checkpoint_sync.py \
  --experiment "$MLFLOW_EXPERIMENT_NAME" \
  --run-uuid "$RUN_UUID" \
  --yaml-config "$YAML_CONFIG" \
  --output-dir "$OUTPUT_DIR" \
  --project-dir "$PROJECT_DIR" \
  --run-name "$EXPERIMENT_VARIANT-$RUN_UUID" \
  --state-path "$SYNC_STATE_PATH" \
  --interval-seconds 180 \
  --stable-seconds 20 \
  --artifact-path checkpoints "${best_args[@]}"


In [ ]:
# %%bash
# # 8. Full Benchmark Test (3 CROHME datasets)
# set -euo pipefail
# cd "$PROJECT_DIR"
# source "$CONDA_DIR/bin/activate" unimumer

# # Tìm checkpoint cuối cùng
# LAST_CKPT=$(ls -td "$OUTPUT_DIR"/checkpoint-* 2>/dev/null | head -1)

# if [[ -z "$LAST_CKPT" ]]; then
#   echo "ERROR: No checkpoint found in $OUTPUT_DIR"
#   exit 1
# fi

# echo "Testing with checkpoint: $LAST_CKPT"
# echo "Running full benchmark on 3 CROHME datasets (3,332 samples)..."
# echo ""

# # Chạy full benchmark
# python scripts/kaggle_full_test.py \
#   --base-model Qwen/Qwen2.5-VL-3B-Instruct \
#   --adapter-path "$LAST_CKPT" \
#   --test-datasets crohme_2014 crohme_2016 crohme_2019 \
#   --backup-dir example_data/backup \
#   --base-results-dir example_data/CROHME/results \
#   --output-dir kaggle_test_results \
#   --project-dir "$PROJECT_DIR" \
#   --batch-size 2

# # In summary
# echo ""
# echo "============================================================"
# echo "                    TEST SUMMARY"
# echo "============================================================"
# for dataset in crohme_2014 crohme_2016 crohme_2019; do
#   echo ""
#   echo "=== $dataset ==="
#   if [[ -f "kaggle_test_results/${dataset}_results.txt" ]]; then
#     cat "kaggle_test_results/${dataset}_results.txt" | grep -E "(Mean Edit Score|BLEU-4|Character Error Rate|Exact Match)" | head -4
#   else
#     echo "Results not found"
#   fi
# done

# echo ""
# echo "============================================================"
# echo "Full comparison table:"
# cat kaggle_test_results/comparison_table.txt

In [ ]:
%%bash
# 9. Upload checkpoint và cấu hình của run này
set -euo pipefail
[[ "$RUN_TRAINING" == 1 ]] || { echo "Bỏ qua upload: RUN_TRAINING=False"; exit 0; }
cd "$PROJECT_DIR"
test -n "$YAML_CONFIG"

"$PYTHON" scripts/dagshub_logger.py upload \
  --experiment "$MLFLOW_EXPERIMENT_NAME" \
  --run-uuid "$RUN_UUID" \
  --config "$YAML_CONFIG" \
  --output-dir "$OUTPUT_DIR" \
  --project-dir "$PROJECT_DIR" \
  --notebook "$NOTEBOOK_PATH"


In [ ]:
# %%bash
# cd /kaggle/working/test-unimer

# echo "Đang ở:"
# pwd

# echo "Branch hiện tại:"
# git branch --show-current

# echo "Trạng thái trước khi pull:"
# git status --short

# echo "Pull code mới nhất:"
# git pull